In [1]:
import pandas as pd
import numpy as np

assets = pd.read_csv(
    "../data/processed/assets_simulated.csv"
)

maintenance = pd.read_csv(
    "../data/processed/maintenance_simulated.csv"
)

history = pd.read_csv(
    "../data/processed/asset_history_simulated.csv"
)

failures = pd.read_csv(
    "../data/processed/failures_simulated.csv"
)

print("Assets:", assets.shape)
print("Maintenance:", maintenance.shape)
print("History:", history.shape)
print("Failures:", failures.shape)

Assets: (1000, 6)
Maintenance: (957, 8)
History: (92000, 10)
Failures: (1048, 7)


In [2]:
import joblib

calibrated_xgboost = joblib.load(
    "../models/calibrated_xgboost.pkl"
)

print("Calibrated XGBoost loaded successfully.")

Calibrated XGBoost loaded successfully.


In [3]:
import pandas as pd
import numpy as np

assets = pd.read_csv(
    "../data/processed/assets_simulated.csv"
)

maintenance = pd.read_csv(
    "../data/processed/maintenance_simulated.csv"
)

history = pd.read_csv(
    "../data/processed/asset_history_simulated.csv"
)

failures = pd.read_csv(
    "../data/processed/failures_simulated.csv"
)

print("Assets:", assets.shape)
print("Maintenance:", maintenance.shape)
print("History:", history.shape)
print("Failures:", failures.shape)

Assets: (1000, 6)
Maintenance: (957, 8)
History: (92000, 10)
Failures: (1048, 7)


In [4]:
import joblib

calibrated_xgboost = joblib.load(
    "../models/calibrated_xgboost.pkl"
)

print("Calibrated XGBoost loaded successfully.")

Calibrated XGBoost loaded successfully.


In [5]:
# Make sure dates are proper datetime values
history["snapshot_date"] = pd.to_datetime(
    history["snapshot_date"]
)

failures["failure_date"] = pd.to_datetime(
    failures["failure_date"]
)

# Sort everything chronologically
history = history.sort_values(
    ["asset_id", "snapshot_date"]
).copy()

failures = failures.sort_values(
    ["asset_id", "failure_date"]
).copy()


# --------------------------------------------------
# 1. Historical failure count
# --------------------------------------------------

failure_counts = []

for _, row in history.iterrows():

    asset_id = row["asset_id"]
    snapshot_date = row["snapshot_date"]

    count = (
        (
            (failures["asset_id"] == asset_id)
            &
            (failures["failure_date"] < snapshot_date)
        )
        .sum()
    )

    failure_counts.append(count)


history["historical_failure_count"] = (
    failure_counts
)


# --------------------------------------------------
# 2. Historical downtime
# --------------------------------------------------

downtime_values = []

for _, row in history.iterrows():

    asset_id = row["asset_id"]
    snapshot_date = row["snapshot_date"]

    previous_failures = failures[
        (failures["asset_id"] == asset_id)
        &
        (failures["failure_date"] < snapshot_date)
    ]

    if len(previous_failures) == 0:
        downtime = 0
    else:
        downtime = previous_failures[
            "downtime_hours"
        ].sum()

    downtime_values.append(downtime)


history["historical_downtime_hours"] = (
    downtime_values
)


# --------------------------------------------------
# 3. Days since last failure
# --------------------------------------------------

days_since_failure = []

for _, row in history.iterrows():

    asset_id = row["asset_id"]
    snapshot_date = row["snapshot_date"]

    previous_failures = failures[
        (failures["asset_id"] == asset_id)
        &
        (failures["failure_date"] < snapshot_date)
    ]

    if len(previous_failures) == 0:

        days = np.nan

    else:

        last_failure_date = (
            previous_failures["failure_date"].max()
        )

        days = (
            snapshot_date - last_failure_date
        ).days

    days_since_failure.append(days)


history["days_since_last_failure"] = (
    days_since_failure
)


# --------------------------------------------------
# Create latest asset state
# --------------------------------------------------

latest_state = (
    history
    .sort_values("snapshot_date")
    .groupby("asset_id")
    .tail(1)
    .copy()
)


print(
    "Latest asset states:",
    latest_state.shape
)

print(
    latest_state[
        [
            "asset_id",
            "snapshot_date",
            "historical_failure_count",
            "historical_downtime_hours",
            "days_since_last_failure"
        ]
    ].head()
)

Latest asset states: (1000, 13)
         asset_id snapshot_date  historical_failure_count  \
91631  ASSET00996    2026-08-01                         0   
91079  ASSET00990    2026-08-01                         0   
91447  ASSET00994    2026-08-01                         0   
91355  ASSET00993    2026-08-01                         4   
91907  ASSET00999    2026-08-01                         1   

       historical_downtime_hours  days_since_last_failure  
91631                   0.000000                      NaN  
91079                   0.000000                      NaN  
91447                   0.000000                      NaN  
91355                  25.150304                    638.0  
91907                   1.568571                    243.0  


In [10]:
# ============================================================
# RECOVERY CELL — restore variables after restarting work
# ============================================================

feature_columns = [
    "asset_age_years",
    "condition_score",
    "criticality",
    "usage_factor",
    "weather_stress",
    "historical_failure_count",
    "historical_downtime_hours",
    "days_since_last_failure"
]

print("feature_columns restored:")
print(feature_columns)

feature_columns restored:
['asset_age_years', 'condition_score', 'criticality', 'usage_factor', 'weather_stress', 'historical_failure_count', 'historical_downtime_hours', 'days_since_last_failure']


In [11]:
X_latest = latest_state[feature_columns]

latest_state["failure_probability"] = (
    calibrated_xgboost.predict_proba(X_latest)[:, 1]
)

print(
    latest_state[
        [
            "asset_id",
            "failure_probability"
        ]
    ]
    .sort_values(
        "failure_probability",
        ascending=False
    )
    .head(10)
)

         asset_id  failure_probability
2115   ASSET00023             0.015974
60351  ASSET00656             0.015590
84731  ASSET00921             0.011155
2943   ASSET00032             0.009979
43515  ASSET00473             0.009901
16375  ASSET00178             0.009714
57315  ASSET00623             0.009672
26863  ASSET00292             0.009497
76911  ASSET00836             0.009466
11407  ASSET00124             0.009151


In [12]:
print(
    latest_state["failure_probability"].describe()
)

count    1000.000000
mean        0.004482
std         0.001372
min         0.003054
25%         0.003542
50%         0.004074
75%         0.005017
max         0.015974
Name: failure_probability, dtype: float64


In [13]:
latest_state["failure_risk"] = (
    latest_state["failure_probability"]
).clip(0, 1)

latest_state["condition_risk"] = (
    1 - latest_state["condition_score"] / 100
).clip(0, 1)

latest_state["criticality_risk"] = (
    latest_state["criticality"] / 10
).clip(0, 1)

latest_state["usage_risk"] = (
    latest_state["usage_factor"] / 1.5
).clip(0, 1)

latest_state["weather_risk"] = (
    latest_state["weather_stress"]
).clip(0, 1)

latest_state["failure_history_risk"] = (
    latest_state["historical_failure_count"] / 5
).clip(0, 1)

print(
    latest_state[
        [
            "asset_id",
            "failure_risk",
            "condition_risk",
            "criticality_risk",
            "usage_risk",
            "weather_risk",
            "failure_history_risk"
        ]
    ].head()
)

         asset_id  failure_risk  condition_risk  criticality_risk  usage_risk  \
91631  ASSET00996      0.005880        0.345351               1.0    0.441443   
91079  ASSET00990      0.006636        0.213560               1.0    0.949822   
91447  ASSET00994      0.005909        0.314897               1.0    0.380034   
91355  ASSET00993      0.003528        0.442772               0.9    0.683019   
91907  ASSET00999      0.004695        0.404183               0.5    0.380034   

       weather_risk  failure_history_risk  
91631      0.468746                   0.0  
91079      0.130947                   0.0  
91447      0.716770                   0.0  
91355      0.763244                   0.8  
91907      0.824726                   0.2  


In [14]:
print("Maintenance columns:")
print(maintenance.columns.tolist())

print("\nSample maintenance records:")
print(maintenance.head())

Maintenance columns:
['task_id', 'asset_id', 'section_id', 'department', 'maintenance_date', 'maintenance_type', 'severity', 'duration_hours']

Sample maintenance records:
      task_id    asset_id  section_id department maintenance_date  \
0  TASK000001  ASSET00002  RTM-VAD-01   TRACTION       2022-01-01   
1  TASK000002  ASSET00002  RTM-VAD-01   TRACTION       2023-06-01   
2  TASK000003  ASSET00003  MTJ-AGC-01        S&T       2019-10-01   
3  TASK000004  ASSET00003  MTJ-AGC-01        S&T       2020-08-01   
4  TASK000005  ASSET00003  MTJ-AGC-01        S&T       2022-02-01   

  maintenance_type  severity  duration_hours  
0       PREVENTIVE         8        2.288423  
1       PREVENTIVE         5        1.277000  
2           REPAIR         3        1.677256  
3           REPAIR         5        2.000026  
4       INSPECTION         3        1.553127  


In [15]:
# Save the latest asset state for the maintenance priority engine

latest_state.to_csv(
    "../data/processed/latest_asset_state.csv",
    index=False
)

print("Saved latest asset state successfully.")
print("Shape:", latest_state.shape)

Saved latest asset state successfully.
Shape: (1000, 20)


In [18]:
# Step 16.10 — Prepare maintenance data

# Load maintenance data
maintenance = pd.read_csv("../data/raw/maintenance.csv")

# Convert available date columns
maintenance["created_date"] = pd.to_datetime(
    maintenance["created_date"],
    errors="coerce"
)

maintenance["due_date"] = pd.to_datetime(
    maintenance["due_date"],
    errors="coerce"
)

# Remove records with invalid creation dates
maintenance = maintenance.dropna(
    subset=["created_date"]
)

# Sort chronologically
maintenance = maintenance.sort_values(
    "created_date"
).reset_index(drop=True)

print("Maintenance records:", maintenance.shape)
print("Maintenance preparation complete.")

print("\nMaintenance columns:")
print(maintenance.columns.tolist())

print("\nSample maintenance records:")
print(maintenance.head())

Maintenance records: (6, 14)
Maintenance preparation complete.

Maintenance columns:
['task_id', 'asset_id', 'section_id', 'department', 'maintenance_type', 'defect_type', 'severity', 'criticality', 'created_date', 'due_date', 'overdue_days', 'estimated_duration', 'required_manpower', 'status']

Sample maintenance records:
   task_id asset_id  section_id   department maintenance_type  \
0  TDMS001   OHE001  GWL-JHS-01     TRACTION           REPAIR   
1   TMS001   TRK001  NDL-MTJ-01  ENGINEERING           REPAIR   
2  SMMS001   SIG001  MTJ-AGC-01          S&T           REPAIR   
3   TMS002   TRK002  NDL-MTJ-02  ENGINEERING       INSPECTION   
4  SMMS002   SIG002  AGC-GWL-01          S&T       INSPECTION   

          defect_type  severity  criticality created_date   due_date  \
0       OHE_INSULATOR         7            9   2026-08-18 2026-09-03   
1          RAIL_CRACK         9           10   2026-08-20 2026-09-01   
2  SIGNAL_DEGRADATION         8            9   2026-08-22 2026-09-04

In [19]:
# Step 16.11 — Calculate maintenance urgency

# Convert important fields to numeric
maintenance["severity"] = pd.to_numeric(
    maintenance["severity"],
    errors="coerce"
)

maintenance["criticality"] = pd.to_numeric(
    maintenance["criticality"],
    errors="coerce"
)

maintenance["overdue_days"] = pd.to_numeric(
    maintenance["overdue_days"],
    errors="coerce"
)

maintenance["estimated_duration"] = pd.to_numeric(
    maintenance["estimated_duration"],
    errors="coerce"
)

# Replace missing numeric values
maintenance["severity"] = maintenance["severity"].fillna(0)
maintenance["criticality"] = maintenance["criticality"].fillna(0)
maintenance["overdue_days"] = maintenance["overdue_days"].fillna(0)

# Normalize the main urgency factors
severity_risk = (
    maintenance["severity"] / 10
).clip(0, 1)

criticality_risk = (
    maintenance["criticality"] / 10
).clip(0, 1)

overdue_risk = (
    maintenance["overdue_days"] / 30
).clip(0, 1)

# Combined maintenance urgency score
maintenance["urgency_score"] = (
    0.40 * severity_risk +
    0.35 * criticality_risk +
    0.25 * overdue_risk
)

# Categorize urgency
maintenance["urgency_category"] = pd.cut(
    maintenance["urgency_score"],
    bins=[-np.inf, 0.25, 0.50, 0.75, np.inf],
    labels=["LOW", "MEDIUM", "HIGH", "CRITICAL"]
)

print("Maintenance urgency calculated successfully.")

print(
    maintenance[
        [
            "task_id",
            "asset_id",
            "department",
            "severity",
            "criticality",
            "overdue_days",
            "urgency_score",
            "urgency_category"
        ]
    ]
    .sort_values("urgency_score", ascending=False)
    .head(10)
)

Maintenance urgency calculated successfully.
   task_id asset_id   department  severity  criticality  overdue_days  \
1   TMS001   TRK001  ENGINEERING         9           10             1   
2  SMMS001   SIG001          S&T         8            9             0   
0  TDMS001   OHE001     TRACTION         7            9             0   
3   TMS002   TRK002  ENGINEERING         4            8             0   
5  TDMS002   OHE002     TRACTION         3            8             0   
4  SMMS002   SIG002          S&T         3            7             0   

   urgency_score urgency_category  
1       0.718333             HIGH  
2       0.635000             HIGH  
0       0.595000             HIGH  
3       0.440000           MEDIUM  
5       0.400000           MEDIUM  
4       0.365000           MEDIUM  


In [20]:
# Step 16.12 — Merge ML failure probability with maintenance tasks

# Get the latest predicted asset risk
asset_risk = latest_state[
    [
        "asset_id",
        "failure_probability"
    ]
].copy()

# Merge failure probability into maintenance tasks
maintenance = maintenance.merge(
    asset_risk,
    on="asset_id",
    how="left"
)

# Assets without a prediction get probability 0
maintenance["failure_probability"] = (
    maintenance["failure_probability"]
    .fillna(0)
)

print("Failure probability merged successfully.")

print(
    maintenance[
        [
            "task_id",
            "asset_id",
            "department",
            "urgency_score",
            "urgency_category",
            "failure_probability"
        ]
    ]
    .sort_values(
        "failure_probability",
        ascending=False
    )
    .head(10)
)

Failure probability merged successfully.
   task_id asset_id   department  urgency_score urgency_category  \
0  TDMS001   OHE001     TRACTION       0.595000             HIGH   
1   TMS001   TRK001  ENGINEERING       0.718333             HIGH   
2  SMMS001   SIG001          S&T       0.635000             HIGH   
3   TMS002   TRK002  ENGINEERING       0.440000           MEDIUM   
4  SMMS002   SIG002          S&T       0.365000           MEDIUM   
5  TDMS002   OHE002     TRACTION       0.400000           MEDIUM   

   failure_probability  
0                  0.0  
1                  0.0  
2                  0.0  
3                  0.0  
4                  0.0  
5                  0.0  


In [21]:
# Step 16.13 — Intelligent Maintenance Priority Score

# Normalize failure probability
failure_risk = (
    maintenance["failure_probability"]
    .clip(0, 1)
)

# Normalize maintenance urgency
urgency_risk = (
    maintenance["urgency_score"]
    .clip(0, 1)
)

# Normalize criticality
criticality_risk = (
    maintenance["criticality"] / 10
).clip(0, 1)

# Normalize overdue days
overdue_risk = (
    maintenance["overdue_days"] / 30
).clip(0, 1)

# Intelligent priority score
maintenance["priority_score"] = (
    0.40 * failure_risk +
    0.30 * urgency_risk +
    0.20 * criticality_risk +
    0.10 * overdue_risk
)

# Priority categories
maintenance["priority_category"] = pd.cut(
    maintenance["priority_score"],
    bins=[-np.inf, 0.25, 0.50, 0.75, np.inf],
    labels=[
        "LOW",
        "MEDIUM",
        "HIGH",
        "CRITICAL"
    ]
)

# Display highest-priority maintenance tasks
priority_view = (
    maintenance[
        [
            "task_id",
            "asset_id",
            "section_id",
            "department",
            "maintenance_type",
            "severity",
            "criticality",
            "overdue_days",
            "failure_probability",
            "urgency_score",
            "priority_score",
            "priority_category"
        ]
    ]
    .sort_values(
        "priority_score",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Intelligent Maintenance Priority Engine completed.")

priority_view.head(15)

Intelligent Maintenance Priority Engine completed.


,task_id,asset_id,section_id,department,maintenance_type,severity,criticality,overdue_days,failure_probability,urgency_score,priority_score,priority_category
0,TMS001,TRK001,NDL-MTJ-01,ENGINEERING,REPAIR,9,10,1,0.0,0.718333,0.418833,MEDIUM
1,SMMS001,SIG001,MTJ-AGC-01,S&T,REPAIR,8,9,0,0.0,0.635000,0.370500,MEDIUM
2,TDMS001,OHE001,GWL-JHS-01,TRACTION,REPAIR,7,9,0,0.0,0.595000,0.358500,MEDIUM
3,TMS002,TRK002,NDL-MTJ-02,ENGINEERING,INSPECTION,4,8,0,0.0,0.440000,0.292000,MEDIUM
4,TDMS002,OHE002,JHS-BINA-01,TRACTION,INSPECTION,3,8,0,0.0,0.400000,0.280000,MEDIUM
5,SMMS002,SIG002,AGC-GWL-01,S&T,INSPECTION,3,7,0,0.0,0.365000,0.249500,LOW


In [22]:
# Step 16.14 — Priority analysis by department and section

# Highest-priority tasks by department
department_priority = (
    maintenance
    .groupby("department", observed=True)
    .agg(
        task_count=("task_id", "count"),
        average_priority=("priority_score", "mean"),
        maximum_priority=("priority_score", "max"),
        critical_tasks=(
            "priority_category",
            lambda x: (x == "CRITICAL").sum()
        )
    )
    .sort_values(
        "average_priority",
        ascending=False
    )
)

print("Priority summary by department:")
display(department_priority)


# Highest-priority tasks by section
section_priority = (
    maintenance
    .groupby("section_id", observed=True)
    .agg(
        task_count=("task_id", "count"),
        average_priority=("priority_score", "mean"),
        maximum_priority=("priority_score", "max"),
        critical_tasks=(
            "priority_category",
            lambda x: (x == "CRITICAL").sum()
        )
    )
    .sort_values(
        "average_priority",
        ascending=False
    )
)

print("\nPriority summary by section:")
display(section_priority)

Priority summary by department:


,task_count,average_priority,maximum_priority,critical_tasks
department,,,,
ENGINEERING,2,0.355417,0.418833,0
TRACTION,2,0.319250,0.358500,0
S&T,2,0.310000,0.370500,0



Priority summary by section:


,task_count,average_priority,maximum_priority,critical_tasks
section_id,,,,
NDL-MTJ-01,1,0.418833,0.418833,0
MTJ-AGC-01,1,0.370500,0.370500,0
GWL-JHS-01,1,0.358500,0.358500,0
NDL-MTJ-02,1,0.292000,0.292000,0
JHS-BINA-01,1,0.280000,0.280000,0
AGC-GWL-01,1,0.249500,0.249500,0


In [23]:
# Step 16.15 — Generate ranked maintenance worklist

ranked_worklist = (
    maintenance[
        [
            "task_id",
            "asset_id",
            "section_id",
            "department",
            "maintenance_type",
            "defect_type",
            "severity",
            "criticality",
            "overdue_days",
            "estimated_duration",
            "required_manpower",
            "status",
            "failure_probability",
            "urgency_score",
            "priority_score",
            "priority_category"
        ]
    ]
    .sort_values(
        ["priority_score", "severity", "overdue_days"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

print("Ranked maintenance worklist created.")

display(ranked_worklist.head(20))

Ranked maintenance worklist created.


,task_id,asset_id,section_id,department,maintenance_type,defect_type,severity,criticality,overdue_days,estimated_duration,required_manpower,status,failure_probability,urgency_score,priority_score,priority_category
0,TMS001,TRK001,NDL-MTJ-01,ENGINEERING,REPAIR,RAIL_CRACK,9,10,1,90,8,PENDING,0.0,0.718333,0.418833,MEDIUM
1,SMMS001,SIG001,MTJ-AGC-01,S&T,REPAIR,SIGNAL_DEGRADATION,8,9,0,45,3,PENDING,0.0,0.635000,0.370500,MEDIUM
2,TDMS001,OHE001,GWL-JHS-01,TRACTION,REPAIR,OHE_INSULATOR,7,9,0,60,5,PENDING,0.0,0.595000,0.358500,MEDIUM
3,TMS002,TRK002,NDL-MTJ-02,ENGINEERING,INSPECTION,TRACK_WEAR,4,8,0,60,5,PENDING,0.0,0.440000,0.292000,MEDIUM
4,TDMS002,OHE002,JHS-BINA-01,TRACTION,INSPECTION,OHE_CHECK,3,8,0,45,4,PENDING,0.0,0.400000,0.280000,MEDIUM
5,SMMS002,SIG002,AGC-GWL-01,S&T,INSPECTION,SIGNAL_CHECK,3,7,0,30,2,PENDING,0.0,0.365000,0.249500,LOW


In [24]:
# Step 16.16 — Assign maintenance priority rank

ranked_worklist.insert(
    0,
    "priority_rank",
    np.arange(1, len(ranked_worklist) + 1)
)

display(
    ranked_worklist.head(20)
)

,priority_rank,task_id,asset_id,section_id,department,maintenance_type,defect_type,severity,criticality,overdue_days,estimated_duration,required_manpower,status,failure_probability,urgency_score,priority_score,priority_category
0,1,TMS001,TRK001,NDL-MTJ-01,ENGINEERING,REPAIR,RAIL_CRACK,9,10,1,90,8,PENDING,0.0,0.718333,0.418833,MEDIUM
1,2,SMMS001,SIG001,MTJ-AGC-01,S&T,REPAIR,SIGNAL_DEGRADATION,8,9,0,45,3,PENDING,0.0,0.635000,0.370500,MEDIUM
2,3,TDMS001,OHE001,GWL-JHS-01,TRACTION,REPAIR,OHE_INSULATOR,7,9,0,60,5,PENDING,0.0,0.595000,0.358500,MEDIUM
3,4,TMS002,TRK002,NDL-MTJ-02,ENGINEERING,INSPECTION,TRACK_WEAR,4,8,0,60,5,PENDING,0.0,0.440000,0.292000,MEDIUM
4,5,TDMS002,OHE002,JHS-BINA-01,TRACTION,INSPECTION,OHE_CHECK,3,8,0,45,4,PENDING,0.0,0.400000,0.280000,MEDIUM
5,6,SMMS002,SIG002,AGC-GWL-01,S&T,INSPECTION,SIGNAL_CHECK,3,7,0,30,2,PENDING,0.0,0.365000,0.249500,LOW


In [25]:
# Step 16.17 — Priority category distribution

priority_distribution = (
    ranked_worklist["priority_category"]
    .value_counts()
    .reindex(
        ["CRITICAL", "HIGH", "MEDIUM", "LOW"],
        fill_value=0
    )
)

print("Maintenance priority distribution:")
print(priority_distribution)

Maintenance priority distribution:
priority_category
CRITICAL    0
HIGH        0
MEDIUM      5
LOW         1
Name: count, dtype: int64


In [26]:
# Step 16.18 — Save ranked maintenance worklist

ranked_worklist.to_csv(
    "../data/predictions/ranked_maintenance_worklist.csv",
    index=False
)

print(
    "Saved ranked worklist successfully:"
)

print(
    "../data/predictions/ranked_maintenance_worklist.csv"
)

Saved ranked worklist successfully:
../data/predictions/ranked_maintenance_worklist.csv
